In [1]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/Project/Xet_Khen_Thuong/Fine-Tune-GLiNER

Mounted at /content/drive
/content/drive/MyDrive/Project/Xet_Khen_Thuong/Fine-Tune-GLiNER


In [2]:
!pip install gliner[training]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 56.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.0/262.0 kB 13.8 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.18.0
    Uninstalling transformers-5.18.0:
      Successfully uninstalled transformers-5.18.0


In [6]:
import json

input_file = "DataTrain/train_data_relations_only.json"
# Dataset NER tiếng Việt cho miền thi đua, khen thưởng.
# 100 mẫu tổng hợp để minh họa/khởi tạo huấn luyện; không phải trích nguyên văn văn bản pháp luật.
# Span NER dùng chỉ số token bắt đầu/kết thúc inclusive: [start_token, end_token, LABEL].
# Nhãn: CA_NHAN, TO_CHUC, CO_QUAN, DOI_TUONG, CHUC_DANH, PHAN_THUONG, DANH_HIEU,
# DIEU_KIEN, HO_SO, VAN_BAN, DIEU_KHOAN, THOI_GIAN, THOI_HAN, SO_LUONG,
# CAP_KHEN_THUONG, LINH_VUC.

ENTITY_LABELS = [
    # 1. Chủ thể
    "CA_NHAN",
    "TO_CHUC",
    "CO_QUAN_NHA_NUOC",
    "NHOM_DOI_TUONG",
    "CHUC_VU",
    # 2. Đối tượng và nội dung khen thưởng
    "DANH_HIEU_THI_DUA",
    "HINH_THUC_KHEN_THUONG",
    "THANH_TICH",
    "CAP_KHEN_THUONG",
    # 3. Điều kiện và tiêu chí xét thưởng
    "DIEU_KIEN",
    "TIEU_CHI",
    "TRUONG_HOP_LOAI_TRU",
    "TIEU_CHUAN",
    # 4. Căn cứ pháp lý
    "VAN_BAN_PHAP_LUAT",
    "DIEU_KHOAN_PHAP_LUAT",
    # 5. Thời gian và định lượng
    "THOI_GIAN",
    "THOI_HAN",
    "SO_LUONG",
    "TY_LE",
    "MUC_TIEN_THUONG",
    # 6. Quy trình xét khen thưởng
    "HO_SO",
    "THU_TUC",
    # 7. Phạm vi áp dụng
    "LINH_VUC",
    "DIA_BAN_AP_DUNG",
]

# Đọc JSON
with open(input_file, "r", encoding="utf-8") as f:
    data = json.load(f)

clean_data = []

for item in data:
    clean_item = {
        "tokenized_text": item["tokenized_text"],
        "ner": item.get("ner", [])
    }

    clean_data.append(clean_item)

# Lưu JSON

print(f"Đã xử lý {len(clean_data)} mẫu")
print(clean_data)

Đã xử lý 195 mẫu
[{'tokenized_text': ['Nguyễn', 'Văn', 'An', 'được', 'tặng', 'Bằng', 'khen', 'của', 'Ủy', 'ban', 'nhân', 'dân', 'thành', 'phố', 'Cần', 'Thơ', '.'], 'ner': [[0, 2, 'CA_NHAN'], [5, 6, 'PHAN_THUONG'], [8, 15, 'CO_QUAN']]}, {'tokenized_text': ['Trường', 'Đại', 'học', 'Cần', 'Thơ', 'đạt', 'danh', 'hiệu', 'Tập', 'thể', 'lao', 'động', 'xuất', 'sắc', '.'], 'ner': [[0, 4, 'TO_CHUC'], [8, 13, 'DANH_HIEU']]}, {'tokenized_text': ['Trần', 'Thị', 'Mai', 'được', 'trao', 'Huân', 'chương', 'Lao', 'động', 'hạng', 'Ba', '.'], 'ner': [[0, 2, 'CA_NHAN'], [5, 10, 'PHAN_THUONG']]}, {'tokenized_text': ['Phạm', 'Quốc', 'Huy', 'được', 'công', 'nhận', 'danh', 'hiệu', 'Chiến', 'sĩ', 'thi', 'đua', 'cơ', 'sở', '.'], 'ner': [[0, 2, 'CA_NHAN'], [8, 13, 'DANH_HIEU']]}, {'tokenized_text': ['Sở', 'Giáo', 'dục', 'và', 'Đào', 'tạo', 'thành', 'phố', 'Cần', 'Thơ', 'được', 'tặng', 'Cờ', 'thi', 'đua', '.'], 'ner': [[0, 9, 'CO_QUAN'], [12, 14, 'PHAN_THUONG']]}, {'tokenized_text': ['Nguyễn', 'Minh', 'Khang', 'đạ

**LOAD GLiNER**

In [ ]:
from gliner import GLiNER

# Load a pretrained model
model = GLiNER.from_pretrained("urchade/gliner_small-v2.1")

# Define your training data
# Train the model
trainer = model.train_model(
    train_dataset=clean_data,
    eval_dataset=clean_data,
    output_dir="/content/drive/MyDrive/Project/Xet_Khen_Thuong/Models",
    max_steps=150,
    learning_rate=5e-5,
    per_device_train_batch_size=8,
)

# Save the trained model
trainer.save_model()

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:190: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1, 'pad_token_id': 0}.
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
10,107.707617
